In [6]:
# 0. Mount Google Drive for Permanent Storage
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import urllib.request
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Download & Extract NIH Malaria Cell Dataset (27,558 images)
url = "https://nih.gov"
if not os.path.exists("cell_images"):
    print("Downloading NIH Malaria Dataset...")
    urllib.request.urlretrieve(url, "cell_images.zip")
    with zipfile.ZipFile("cell_images.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download & Extraction Complete!")

data_dir = "cell_images/cell_images"
IMG_SIZE = (96, 96)  # Compact resolution for fast edge inference
BATCH_SIZE = 64

# 2. Load Train & Validation Splits (80/20)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")  # ['Parasitized', 'Uninfected']

# Optimize data pipeline
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Build Lightweight Edge-Ready MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Fast transfer learning

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train Model (5 Epochs - takes ~1-2 minutes on Colab GPU)
print("\nTraining Edge Diagnostic Model...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 5. Evaluate Clinical Metrics (Sensitivity, Specificity, Accuracy)
print("\nEvaluating Validation Set...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_true = np.array(y_true)
# Note: In Keras alphabetical order, 0 = Parasitized, 1 = Uninfected
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tp, fn, fp, tn = cm.ravel()  # Class 0 (Parasitized) is positive case

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100

# 6. Convert to TensorFlow Lite & Save Directly to Google Drive
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Post-training quantization
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

# Benchmark CPU Inference Latency
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
# Warmup
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()

# Timed 100 runs on CPU
latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)

# 7. Print & Save Final Abstract Metrics Summary to Drive
summary_text = (
    "=============================================\n"
    "RESULTS FOR YOUR CONFERENCE ABSTRACT\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)

print("\n" + summary_text)

report_path = os.path.join(save_dir, "abstract_metrics.txt")
with open(report_path, "w") as f:
    f.write(summary_text)

print(f"Saved model and metrics permanently to: {save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


HTTPError: HTTP Error 403: Forbidden

In [ ]:
# 0. Mount Google Drive for Permanent Storage
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Load Official NIH Malaria Dataset via TensorFlow Datasets (27,558 images)
print("Loading NIH Malaria Dataset...")
(raw_train, raw_val), ds_info = tfds.load(
    'malaria',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)
print(f"Dataset Loaded! Classes: {ds_info.features['label'].names}")

IMG_SIZE = (96, 96)
BATCH_SIZE = 64

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train.map(preprocess, num_parallel_calls=AUTOTUNE).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = raw_val.map(preprocess, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)

# 2. Build Lightweight Edge-Ready MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 3. Train Model (5 Epochs - takes ~1-2 minutes on Colab GPU)
print("\nTraining Edge Diagnostic Model...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 4. Evaluate Clinical Metrics (Sensitivity, Specificity, Accuracy)
print("\nEvaluating Validation Set...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_true = np.array(y_true)
# In TFDS malaria: 0 = parasitized, 1 = uninfected
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tp, fn, fp, tn = cm.ravel()

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100

# 5. Convert to TensorFlow Lite & Save Directly to Google Drive
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

# Benchmark CPU Inference Latency
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()

latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)

# 6. Print & Save Final Abstract Metrics Summary to Drive
summary_text = (
    "=============================================\n"
    "RESULTS FOR YOUR CONFERENCE ABSTRACT\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)

print("\n" + summary_text)

report_path = os.path.join(save_dir, "abstract_metrics.txt")
with open(report_path, "w") as f:
    f.write(summary_text)

print(f"Saved model and metrics permanently to: {save_dir}")

In [ ]:
# 0. Mount Google Drive for Permanent Storage
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import zipfile
import subprocess
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Download Official NIH Malaria Dataset via curl with Browser User-Agent
url = "https://nih.gov"
if not os.path.exists("cell_images"):
    print("Downloading NIH Malaria Dataset (337 MB)...")
    subprocess.run([
        "curl", "-L", "-A", "Mozilla/5.0",
        url, "-o", "cell_images.zip"
    ], check=True)
    print("Extracting images...")
    with zipfile.ZipFile("cell_images.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download & Extraction Complete!")

data_dir = "cell_images/cell_images"
IMG_SIZE = (96, 96)
BATCH_SIZE = 64

# 2. Load Train & Validation Splits (80/20)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")  # ['Parasitized', 'Uninfected']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Build Lightweight Edge-Ready MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train Model (5 Epochs - takes ~1-2 minutes on Colab GPU)
print("\nTraining Edge Diagnostic Model...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 5. Evaluate Clinical Metrics (Sensitivity, Specificity, Accuracy)
print("\nEvaluating Validation Set...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_true = np.array(y_true)
# Note: In Keras alphabetical order, 0 = Parasitized, 1 = Uninfected
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tp, fn, fp, tn = cm.ravel()

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100

# 6. Convert to TensorFlow Lite & Save Directly to Google Drive
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

# Benchmark CPU Inference Latency
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details['index'], dummy_input)
    interpreter.invoke()

latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)

# 7. Print & Save Final Abstract Metrics Summary to Drive
summary_text = (
    "=============================================\n"
    "RESULTS FOR YOUR CONFERENCE ABSTRACT\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)

print("\n" + summary_text)

report_path = os.path.join(save_dir, "abstract_metrics.txt")
with open(report_path, "w") as f:
    f.write(summary_text)

print(f"Saved model and metrics permanently to: {save_dir}")

In [ ]:
# 0. Mount Google Drive for Permanent Storage
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix
from datasets import load_dataset

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Load Official NIH Malaria Dataset via Hugging Face Mirror (27,558 images)
print("Downloading NIH Malaria Dataset from Hugging Face...")
ds = load_dataset("hibana2077/malaria-cell-images", split="train")
ds = ds.train_test_split(test_size=0.2, seed=42)
print(f"Dataset Loaded! Train: {len(ds['train'])}, Val: {len(ds['test'])}")

IMG_SIZE = (96, 96)
BATCH_SIZE = 64

# 2. Convert to Fast TensorFlow Data Pipeline
def generator(hf_split):
    for item in hf_split:
        img = item["image"].convert("RGB").resize(IMG_SIZE)
        # Labels: 0 = Parasitized, 1 = Uninfected
        yield np.array(img, dtype=np.float32), item["label"]

train_ds = tf.data.Dataset.from_generator(
    lambda: generator(ds["train"]),
    output_signature=(
        tf.TensorSpec(shape=(96, 96, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    lambda: generator(ds["test"]),
    output_signature=(
        tf.TensorSpec(shape=(96, 96, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 3. Build Lightweight Edge-Ready MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train Model (5 Epochs - takes ~1-2 minutes on Colab GPU)
print("\nTraining Edge Diagnostic Model...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 5. Evaluate Clinical Metrics (Sensitivity, Specificity, Accuracy)
print("\nEvaluating Validation Set...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tp, fn, fp, tn = cm.ravel()

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100

# 6. Convert to TensorFlow Lite & Save Directly to Google Drive
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

# Benchmark CPU Inference Latency
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details['index'], dummy_input)
    interpreter.invoke()

latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)

# 7. Print & Save Final Abstract Metrics Summary to Drive
summary_text = (
    "=============================================\n"
    "RESULTS FOR YOUR CONFERENCE ABSTRACT\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)

print("\n" + summary_text)

report_path = os.path.join(save_dir, "abstract_metrics.txt")
with open(report_path, "w") as f:
    f.write(summary_text)

print(f"Saved model and metrics permanently to: {save_dir}")

In [ ]:
# 0. Mount Google Drive for Permanent Storage
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix
import kagglehub

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Download Verified NIH Malaria Dataset via KaggleHub (No login required)
print("Downloading NIH Malaria Dataset via KaggleHub...")
dataset_path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")

# Locate the folder containing 'Parasitized' and 'Uninfected'
data_dir = None
for root, dirs, files in os.walk(dataset_path):
    if "Parasitized" in dirs and "Uninfected" in dirs:
        data_dir = root
        break

print(f"Dataset verified at: {data_dir}")

IMG_SIZE = (96, 96)
BATCH_SIZE = 64

# 2. Load Train & Validation Splits (80/20)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")  # ['Parasitized', 'Uninfected']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Build Lightweight Edge-Ready MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train Model (5 Epochs - takes ~1-2 minutes on Colab GPU)
print("\nTraining Edge Diagnostic Model...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 5. Evaluate Clinical Metrics (Sensitivity, Specificity, Accuracy)
print("\nEvaluating Validation Set...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_true = np.array(y_true)
# In Keras alphabetical order: 0 = Parasitized, 1 = Uninfected
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tp, fn, fp, tn = cm.ravel()

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100

# 6. Convert to TensorFlow Lite & Save Directly to Google Drive
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

# Benchmark CPU Inference Latency
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details['index'], dummy_input)
    interpreter.invoke()

latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)

# 7. Print & Save Final Abstract Metrics Summary to Drive
summary_text = (
    "=============================================\n"
    "RESULTS FOR YOUR CONFERENCE ABSTRACT\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)

print("\n" + summary_text)

report_path = os.path.join(save_dir, "abstract_metrics.txt")
with open(report_path, "w") as f:
    f.write(summary_text)

print(f"Saved model and metrics permanently to: {save_dir}")

In [1]:
# 0. Mount Google Drive for Permanent Storage
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix
import kagglehub

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Locate Verified NIH Malaria Dataset via KaggleHub
print("Locating NIH Malaria Dataset...")
dataset_path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")

data_dir = None
for root, dirs, files in os.walk(dataset_path):
    if "Parasitized" in dirs and "Uninfected" in dirs:
        data_dir = root
        break

print(f"Dataset verified at: {data_dir}")

IMG_SIZE = (96, 96)
BATCH_SIZE = 64

# 2. Load ONLY the 2 True Classes (Ignores duplicate 'cell_images' subfolder)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    class_names=['Parasitized', 'Uninfected'],
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    class_names=['Parasitized', 'Uninfected'],
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print(f"Verified Classes: {train_ds.class_names}")  # Strictly ['Parasitized', 'Uninfected']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Build Lightweight Edge-Ready MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train Model (5 Epochs - ~90 seconds on T4 GPU)
print("\nTraining Edge Diagnostic Model...")
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# 5. Evaluate Clinical Metrics (Sensitivity, Specificity, Accuracy)
print("\nEvaluating Validation Set...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_true = np.array(y_true)
# Label 0 = Parasitized, Label 1 = Uninfected
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tp, fn, fp, tn = cm.ravel()

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100

# 6. Convert to TensorFlow Lite & Save Directly to Google Drive
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

# Benchmark CPU Inference Latency
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()

latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)

# 7. Print & Save Final Abstract Metrics Summary to Drive
summary_text = (
    "=============================================\n"
    "RESULTS FOR YOUR CONFERENCE ABSTRACT\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)

print("\n" + summary_text)

report_path = os.path.join(save_dir, "abstract_metrics.txt")
with open(report_path, "w") as f:
    f.write(summary_text)

print(f"Saved model and metrics permanently to: {save_dir}")

Mounted at /content/drive
Locating NIH Malaria Dataset...
Using Colab cache for faster access to the 'cell-images-for-detecting-malaria' dataset.
Dataset verified at: /kaggle/input/cell-images-for-detecting-malaria/cell_images
Found 27558 files belonging to 2 classes.
Using 22047 files for training.
Found 27558 files belonging to 2 classes.
Using 5511 files for validation.
Verified Classes: ['Parasitized', 'Uninfected']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training Edge Diagnostic Model...
Epoch 1/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 131s 145ms/step - accuracy: 0.8697 - loss: 0.3002 - val_accuracy: 0.9254 - val_loss: 0.2161
Epoch 2/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9150 - loss: 0.2208 - val_accuracy: 0.9296 - val_loss: 0.1922
Epoch 3/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9195 - loss: 0.2100 - val_accuracy: 0.9329 - val_loss: 0.1879
Epoch 4/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9229 - loss: 0.2027 - val_accuracy: 0.9

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



RESULTS FOR YOUR CONFERENCE ABSTRACT
Total Validation Samples : 5511
Overall Accuracy         : 93.49%
Sensitivity (Recall)     : 92.31%
Specificity              : 94.63%
Quantized Model Size     : 2.41 MB
Avg CPU Inference Time   : 4.07 ms/cell

Saved model and metrics permanently to: /content/drive/MyDrive/Malaria_Edge_Project


In [2]:
# 0. Install SHAP (takes ~5 seconds) & Mount Google Drive
!pip install -q shap

from google.colab import drive
drive.mount('/content/drive')

import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

save_dir = "/content/drive/MyDrive/Maternal_XAI_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Download Verified UCI Maternal Health Risk Dataset (1,014 rural clinic records)
print("Downloading UCI Maternal Health Risk Dataset...")
dataset_path = kagglehub.dataset_download("joebeachcapital/maternal-health-risk")
csv_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(".csv")]

df = pd.read_csv(csv_files)
print(f"Loaded {len(df)} patient records with columns: {list(df.columns)}")

# 2. Preprocess Features & Target (Low Risk = 0, Mid Risk = 1, High Risk = 2)
X = df.drop("RiskLevel", axis=1)
label_mapping = {"low risk": 0, "mid risk": 1, "high risk": 2}
y = df["RiskLevel"].str.lower().map(label_mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train & Benchmark Random Forest vs. XGBoost
models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, eval_metric="mlogloss")
}

results_summary = []

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds) * 100
    macro_f1 = f1_score(y_test, preds, average="macro") * 100
    high_risk_recall = recall_score(y_test, preds, labels=, average="macro") * 100

    # Measure single-patient CPU inference speed
    sample = X_test.iloc[]
    start = time.perf_counter()
    for _ in range(500):
        clf.predict(sample)
    latency_ms = ((time.perf_counter() - start) / 500) * 1000

    results_summary.append((name, acc, high_risk_recall, macro_f1, latency_ms))

# Save the best model (XGBoost) & measure size
best_model = models["XGBoost"]
model_path = os.path.join(save_dir, "maternal_xgb_triage.joblib")
joblib.dump(best_model, model_path)
model_size_kb = os.path.getsize(model_path) / 1024

# 4. Generate Explainable AI (SHAP) Clinical Feature Importance Plot
print("Generating SHAP Explainability Chart...")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, plot_type="bar", class_names=["Low Risk", "Mid Risk", "High Risk"], show=False)
plt.title("Clinical Vital Signs Driving Maternal Risk Predictions (SHAP)")
plt.tight_layout()
shap_plot_path = os.path.join(save_dir, "shap_feature_importance.png")
plt.savefig(shap_plot_path, dpi=300)
plt.close()

# 5. Print & Save Final Metrics Summary to Google Drive
lines = [
    "=======================================================",
    "MATERNAL HEALTH XAI TRIAGE - BENCHMARK RESULTS",
    "=======================================================",
    f"Dataset Size             : {len(df)} rural clinic records (Test: {len(X_test)})",
]
for name, acc, hr_rec, f1, lat in results_summary:
    lines.append(f"{name:<16} -> Acc: {acc:.2f}% | High-Risk Recall: {hr_rec:.2f}% | F1: {f1:.2f}% | Latency: {lat:.3f} ms")

lines.extend([
    f"Deployable Model Size    : {model_size_kb:.1f} KB",
    f"SHAP Visual Chart Saved  : {shap_plot_path}",
    "======================================================="
])

report_text = "\n".join(lines)
print("\n" + report_text)

with open(os.path.join(save_dir, "maternal_metrics.txt"), "w") as f:
    f.write(report_text)

print(f"\nAll files saved permanently to: {save_dir}")

SyntaxError: expected argument value expression (2035516543.py, line 54)

In [3]:
# 0. Install SHAP (takes ~5 seconds) & Mount Google Drive
!pip install -q shap

from google.colab import drive
drive.mount('/content/drive')

import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

save_dir = "/content/drive/MyDrive/Maternal_XAI_Project"
os.makedirs(save_dir, exist_ok=True)

# 1. Download Verified UCI Maternal Health Risk Dataset (1,014 rural clinic records)
print("Downloading UCI Maternal Health Risk Dataset...")
dataset_path = kagglehub.dataset_download("joebeachcapital/maternal-health-risk")
csv_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(".csv")]

df = pd.read_csv(csv_files[0])
print(f"Loaded {len(df)} patient records with columns: {list(df.columns)}")

# 2. Preprocess Features & Target (Low Risk = 0, Mid Risk = 1, High Risk = 2)
X = df.drop("RiskLevel", axis=1)
label_mapping = {"low risk": 0, "mid risk": 1, "high risk": 2}
y = df["RiskLevel"].str.lower().map(label_mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train & Benchmark Random Forest vs. XGBoost
models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, eval_metric="mlogloss")
}

results_summary = []

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds) * 100
    macro_f1 = f1_score(y_test, preds, average="macro") * 100
    high_risk_recall = recall_score(y_test, preds, labels=[2], average="macro") * 100

    # Measure single-patient CPU inference speed
    sample = X_test.iloc[[0]]
    start = time.perf_counter()
    for _ in range(500):
        clf.predict(sample)
    latency_ms = ((time.perf_counter() - start) / 500) * 1000

    results_summary.append((name, acc, high_risk_recall, macro_f1, latency_ms))

# Save the best model (XGBoost) & measure size
best_model = models["XGBoost"]
model_path = os.path.join(save_dir, "maternal_xgb_triage.joblib")
joblib.dump(best_model, model_path)
model_size_kb = os.path.getsize(model_path) / 1024

# 4. Generate Explainable AI (SHAP) Clinical Feature Importance Plot
print("Generating SHAP Explainability Chart...")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, plot_type="bar", class_names=["Low Risk", "Mid Risk", "High Risk"], show=False)
plt.title("Clinical Vital Signs Driving Maternal Risk Predictions (SHAP)")
plt.tight_layout()
shap_plot_path = os.path.join(save_dir, "shap_feature_importance.png")
plt.savefig(shap_plot_path, dpi=300)
plt.close()

# 5. Print & Save Final Metrics Summary to Google Drive
lines = [
    "=======================================================",
    "MATERNAL HEALTH XAI TRIAGE - BENCHMARK RESULTS",
    "=======================================================",
    f"Dataset Size             : {len(df)} rural clinic records (Test: {len(X_test)})",
]
for name, acc, hr_rec, f1, lat in results_summary:
    lines.append(f"{name:<16} -> Acc: {acc:.2f}% | High-Risk Recall: {hr_rec:.2f}% | F1: {f1:.2f}% | Latency: {lat:.3f} ms")

lines.extend([
    f"Deployable Model Size    : {model_size_kb:.1f} KB",
    f"SHAP Visual Chart Saved  : {shap_plot_path}",
    "======================================================="
])

report_text = "\n".join(lines)
print("\n" + report_text)

with open(os.path.join(save_dir, "maternal_metrics.txt"), "w") as f:
    f.write(report_text)

print(f"\nAll files saved permanently to: {save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 3.77k/3.77k [00:00<00:00, 5.46MB/s]

Extracting files...
Loaded 1014 patient records with columns: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']


Generating SHAP Explainability Chart...

MATERNAL HEALTH XAI TRIAGE - BENCHMARK RESULTS
Dataset Size             : 1014 rural clinic records (Test: 203)
Random Forest    -> Acc: 84.73% | High-Risk Recall: 96.36% | F1: 85.17% | Latency: 10.136 ms
XGBoost          -> Acc: 81.28% | High-Risk Recall: 94.55% | F1: 81.95% | Latency: 1.627 ms
Deployable Model Size    : 700.2 KB
SHAP Visual Chart Saved  : /content/drive/MyDrive/Maternal_XAI_Project/shap_feature_importance.png

All files saved permanently to: /content/drive/MyDrive/Maternal_XAI_Project


In [4]:
with open("/content/drive/MyDrive/Maternal_XAI_Project/maternal_metrics.txt", "r") as f:
    print(f.read())

MATERNAL HEALTH XAI TRIAGE - BENCHMARK RESULTS
Dataset Size             : 1014 rural clinic records (Test: 203)
Random Forest    -> Acc: 84.73% | High-Risk Recall: 96.36% | F1: 85.17% | Latency: 10.136 ms
XGBoost          -> Acc: 81.28% | High-Risk Recall: 94.55% | F1: 81.95% | Latency: 1.627 ms
Deployable Model Size    : 700.2 KB
SHAP Visual Chart Saved  : /content/drive/MyDrive/Maternal_XAI_Project/shap_feature_importance.png
